# Notebook 06 — Signal Decay and Structural Break Check

**Across the Water: French Nuclear Unplanned Outages and GB Power Prices**

**Conditional on Notebook 02 DA gate passing.**

Tests whether the causal edge has diminished over time — evidence the market
has learned to exploit the signal.

---

### Pre-registered tests

1. **Rolling 2-year β** — primary test. Coefficient estimated on rolling 2-year
   window, plotted year-by-year. Declining β → arbitrage.
   
2. **Sharpe secondary only.** Sharpe conflates signal decay with volatility regime
   changes (TTF shock post-2022, French fleet variance). A stable β with rising
   Sharpe indicates intact signal in a more volatile market — different from decay.

3. **Structural break — Chow test** at 2022-01-01 (TTF price shock period).

---

### Interpretation guide

| Pattern | Interpretation |
|---|---|
| β stable or rising | Signal intact; market not efficiently exploiting it |
| β declining monotonically | Signal being arbitraged away |
| β declining → zero from 2022 | TTF shock changed transmission mechanism |
| β stable, Sharpe rising | Market more volatile; signal intact |


> 📋 **NOT RUN** — Pre-registered but not executed.
> The DA hard gate (Notebook 02) failed before this gate was reached.
> Published unmodified to demonstrate pre-registration discipline.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats as scipy_stats

from src import log, load, DATA_DIR, MAIN_START, MAIN_END
from src.utils import add_season_col

pd.set_option("display.float_format", "{:.4f}".format)
print("Setup complete.")


---
## 1. Rebuild Panel

In [ ]:
gb_da = load("elexon_da_prices")[["gb_da_price"]]
gb_da.index = pd.to_datetime(gb_da.index, utc=True)

entso = load("entso_unavailability")
entso.index = pd.to_datetime(entso.index, utc=True)

def daily_outage_mw(df, t):
    s = df[df["outage_type"]==t]
    return s.groupby(s.index.normalize())["unavailable_mw"].sum().rename(f"{t}_outage_mw")

neso = load("neso_demand")
neso.index = pd.to_datetime(neso.index, utc=True)
neso_d = neso.resample("D").mean()
neso_d["wind_forecast_error"]   = neso_d.get("wind_actual_mw", np.nan) - neso_d.get("wind_forecast_mw", np.nan)
neso_d["demand_forecast_error"] = neso_d.get("demand_actual_mw", np.nan) - neso_d.get("demand_forecast_mw", np.nan)
ifa_cols = [c for c in ["ifa1_flow_mw","ifa2_flow_mw"] if c in neso_d.columns]
neso_d["ifa_flow_mw"] = neso_d[ifa_cols].sum(axis=1) if ifa_cols else np.nan

fr_temp = load("fr_temperature")[["fr_temp_deviation"]]
fr_temp.index = pd.to_datetime(fr_temp.index, utc=True)

try:
    ttf = load("ttf_spot")[["ttf_spot_eur_mwh"]]
    ttf.index = pd.to_datetime(ttf.index, utc=True)
except FileNotFoundError:
    ttf = pd.DataFrame(columns=["ttf_spot_eur_mwh"])

panel = gb_da.copy()
panel = panel.join(daily_outage_mw(entso,"unplanned"), how="left")
panel = panel.join(daily_outage_mw(entso,"planned"),   how="left")
panel = panel.join(neso_d[["wind_forecast_error","demand_forecast_error","ifa_flow_mw"]], how="left")
panel = panel.join(fr_temp, how="left")
if not ttf.empty:
    panel = panel.join(ttf, how="left")
else:
    panel["ttf_spot_eur_mwh"] = np.nan

panel["unplanned_outage_mw"] = panel["unplanned_outage_mw"].fillna(0)
panel["planned_outage_mw"]   = panel["planned_outage_mw"].fillna(0)
panel = panel[panel.index >= pd.Timestamp(MAIN_START, tz="UTC")]
panel = add_season_col(panel)
panel["year"] = panel.index.year

print(f"Panel: {len(panel)} rows")


---
## 2. Rolling 2-year β (Primary Test)

In [ ]:
CONTROLS = [
    "planned_outage_mw", "wind_forecast_error", "demand_forecast_error",
    "ifa_flow_mw", "fr_temp_deviation", "ttf_spot_eur_mwh",
]
WINDOW_DAYS = 365 * 2  # 2-year rolling window

def estimate_beta(sub_panel):
    """Estimate unplanned outage β on a sub-panel. Returns (beta, se, n)."""
    available = [c for c in CONTROLS if c in sub_panel.columns]
    season_d  = pd.get_dummies(sub_panel["season"], prefix="s", drop_first=True)
    
    X = pd.concat([sub_panel[["unplanned_outage_mw"] + available], season_d], axis=1).astype(float)
    X = sm.add_constant(X)
    y = sub_panel["gb_da_price"].astype(float)
    
    mask = X.notna().all(axis=1) & y.notna()
    if mask.sum() < 60:
        return float("nan"), float("nan"), 0
    
    try:
        model = sm.OLS(y[mask], X[mask]).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
        return (
            model.params.get("unplanned_outage_mw", float("nan")) * 1000,  # £/MWh per GW
            model.bse.get("unplanned_outage_mw", float("nan")) * 1000,
            int(model.nobs),
        )
    except Exception:
        return float("nan"), float("nan"), 0

# Roll by year-end anchors
year_ends = pd.date_range("2022-01-01", panel.index.max(), freq="YE", tz="UTC")

rolling_results = []
for end_date in year_ends:
    start_date = end_date - pd.Timedelta(days=WINDOW_DAYS)
    sub = panel[(panel.index >= start_date) & (panel.index <= end_date)]
    beta, se, n = estimate_beta(sub)
    rolling_results.append({
        "window_end": end_date,
        "beta_gw":    beta,
        "se_gw":      se,
        "n":          n,
    })
    print(f"  Window end {end_date.date()}: β={beta:.4f} £/MWh per GW  (n={n})")

rolling_df = pd.DataFrame(rolling_results).set_index("window_end")
rolling_df.to_parquet(DATA_DIR / "rolling_beta.parquet")
print(f"\nSaved rolling_beta.parquet")


In [ ]:
# ── Interpretation ────────────────────────────────────────────────────────────
if len(rolling_df.dropna()) >= 2:
    betas = rolling_df["beta_gw"].dropna()
    first_half = betas.iloc[:len(betas)//2].mean()
    second_half = betas.iloc[len(betas)//2:].mean()
    
    print("\nROLLING β STABILITY:")
    print(f"  First-half mean β:  {first_half:.4f} £/MWh per GW")
    print(f"  Second-half mean β: {second_half:.4f} £/MWh per GW")
    print()
    
    if second_half < first_half * 0.5:
        print("⚠️  β has declined by >50% — consistent with signal decay (arbitrage).")
        print("   This is the expected trajectory for a well-known mechanism.")
    elif second_half > first_half:
        print("✅  β is stable or increasing — signal not showing systematic decay.")
    else:
        print("→  β showing modest decline — monitor with future data.")


---
## 3. Structural Break Test (Chow Test at 2022-01-01)

In [ ]:
BREAK_DATE = pd.Timestamp("2022-01-01", tz="UTC")

pre  = panel[panel.index < BREAK_DATE]
post = panel[panel.index >= BREAK_DATE]

def simple_beta(sub):
    """Quick OLS β estimate for Chow test (simpler specification)."""
    X = sm.add_constant(sub[["unplanned_outage_mw"]].astype(float))
    y = sub["gb_da_price"].astype(float)
    mask = X.notna().all(axis=1) & y.notna()
    if mask.sum() < 30:
        return None
    return sm.OLS(y[mask], X[mask]).fit()

model_pre  = simple_beta(pre)
model_post = simple_beta(post)
model_full = simple_beta(panel)

if model_pre and model_post and model_full:
    # Chow F-statistic
    rss_full = model_pre.ssr + model_post.ssr
    rss_restricted = model_full.ssr
    k = 2   # number of parameters (const + beta)
    n = model_full.nobs
    
    F = ((rss_restricted - rss_full) / k) / (rss_full / (n - 2*k))
    p_chow = 1 - scipy_stats.f.cdf(F, k, n - 2*k)
    
    beta_pre  = model_pre.params.get("unplanned_outage_mw", float("nan")) * 1000
    beta_post = model_post.params.get("unplanned_outage_mw", float("nan")) * 1000
    
    print("CHOW TEST — Structural break at 2022-01-01 (TTF shock)")
    print(f"  Pre-2022  β: {beta_pre:.4f} £/MWh per GW  (n={model_pre.nobs:.0f})")
    print(f"  Post-2022 β: {beta_post:.4f} £/MWh per GW  (n={model_post.nobs:.0f})")
    print(f"  Chow F:      {F:.3f}")
    print(f"  p-value:     {p_chow:.4f}")
    print()
    
    if p_chow < 0.05:
        print("⚠️  Significant structural break detected at 2022.")
        print("   TTF shock / high-price regime appears to have changed the coefficient.")
        print("   Report both sub-period estimates and note the regime dependence.")
    else:
        print("✅  No significant structural break at 2022. β is stable across regimes.")
else:
    print("Insufficient data for Chow test.")


---
## 4. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Rolling β ─────────────────────────────────────────────────────────────────
ax = axes[0]
rd = rolling_df.dropna(subset=["beta_gw"])
if len(rd) > 0:
    ax.plot(rd.index, rd["beta_gw"], color="steelblue", lw=2, marker="o", ms=6, label="Rolling 2yr β")
    ax.fill_between(
        rd.index,
        rd["beta_gw"] - 1.96 * rd["se_gw"],
        rd["beta_gw"] + 1.96 * rd["se_gw"],
        alpha=0.2, color="steelblue"
    )
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.axhline(2, color="grey",  lw=0.8, ls=":",  label="£2/MWh gate threshold")
ax.set_title("Rolling 2-Year β (Primary Decay Test)")
ax.set_ylabel("β: £/MWh per GW")
ax.legend()
ax.set_xlabel("Window end date")

# ── Annual mean GB DA price: event vs control ──────────────────────────────────
ax = axes[1]
panel["is_event"] = (panel["unplanned_outage_mw"] > 0).astype(int)
annual = panel.groupby(["year", "is_event"])["gb_da_price"].mean().unstack()
if 0 in annual.columns and 1 in annual.columns:
    ax.bar(annual.index - 0.2, annual[0], 0.35, color="steelblue", alpha=0.8, label="Control")
    ax.bar(annual.index + 0.2, annual[1], 0.35, color="tomato",    alpha=0.8, label="Event")
ax.set_title("Mean Annual GB DA Price: Event vs Control")
ax.set_ylabel("£/MWh")
ax.set_xlabel("Year")
ax.legend()

plt.suptitle("Notebook 06 — Signal Decay and Structural Break", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "plot_06_decay.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Summary

| Test | Result |
|---|---|
| Rolling 2-yr β trend | See output |
| β stability classification | See output |
| Chow test at 2022-01-01 | See output |

**Interpretation:**
- If β is declining toward zero → signal is being arbitraged away; no new positions advised.
- If β is stable → signal may persist; continue monitoring.
- If structural break at 2022 → regime-dependent trading rule required.

---

*End of Across the Water notebook sequence.*
